# SOP vs TTNO Operator Baseline

This notebook compares an ML-MCTDH-style flat sum-of-products operator baseline with the existing compressed TTNO representation on the same TTNS tree, local basis, and symbolic Hamiltonian.


## 1. Purpose

The junction benchmark follows the operator/tree construction pattern of `../ttns-test/junction_zt_hubbard.py`, the script used for the paper calculations. The external script is not imported directly because it executes argument parsing, logging setup, TTNO construction, TTNS expansion, and time evolution at module import time.

The SOP baseline represents the traditional MCTDH / ML-MCTDH operator layer: a Hamiltonian is kept as a flat list of product terms and each product term is applied independently. It is not a full ML-MCTDH propagator or SPF-equation implementation.

TTNO and SOP use the same symbolic Hamiltonian, the same tree topology, and the same local basis. The comparison therefore targets operator representation efficiency. The notebook focuses on two experimentally useful axes: basis-size growth in the Hubbard junction model and product-operator count growth in an artificial shared-structure stress test. TFD is not treated as a separate TTNO-construction question here because it is a Hamiltonian preparation choice, not a different low-level TTNO algorithm.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
RESULT_DIR = Path('benchmarks/results')
csv_files = ['lead_scaling.csv', 'phonon_scaling.csv', 'shared_structure_scaling.csv']
frames = []
for name in csv_files:
    path = RESULT_DIR / name
    if path.exists():
        df_part = pd.read_csv(path)
        df_part['source_csv'] = name
        frames.append(df_part)
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
df.head()


## 2. Correctness Checks

Some benchmark points exceed 1e-08. Max relative apply error = 1.616e-08; max expectation error = 2.776e-16. See the raw table for details.

In [ ]:
if not df.empty:
    display(df[['case', 'n_lead', 'n_phonon', 'relative_apply_error', 'expectation_error', 'current_relative_apply_error', 'status']])
else:
    print('No benchmark CSV files found.')


## 3. Raw Benchmark Table

This table keeps the quantities needed to audit the comparison: term count, TTNO bond dimension, separated construction/apply timings, speedup, and numerical error. `sop_apply_speedup` means `SOP apply time / TTNO apply time`.


In [ ]:
cols = ['case', 'n_lead', 'n_phonon', 'n_sites', 'local_dim_summary', 'sop_n_terms', 'ttno_max_bond_dim', 'ttno_total_tensor_elements', 'sop_build_time_median', 'ttno_build_time_median', 'sop_apply_time_median', 'ttno_apply_time_median', 'sop_apply_speedup', 'relative_apply_error', 'status']
if not df.empty:
    display(df[[c for c in cols if c in df.columns]])


## 4. Lead Basis Scaling

These figures vary `n_lead`, matching the `--nemode` parameter in `junction_zt_hubbard.py`. Each increment adds four electronic lead sites: left/right electrodes and spin-up/spin-down channels. This is the basis-size growth most directly relevant to the published molecular junction setup.

The SOP-term plot shows how many flattened product terms the baseline must traverse. The TTNO-bond plot shows whether the compressed operator network grows mildly or sharply. The apply-time plot is the actual cost of `H psi`, while the construction-time plot is separated so TTNO setup overhead is not confused with contraction cost. The speedup plot summarizes the apply-time crossover.

![Lead SOP terms](fig_lead_terms.png)

![Lead TTNO bond](fig_lead_bond.png)

![Lead apply time](fig_lead_apply_time.png)

![Lead construction time](fig_lead_build_time.png)

![Lead speedup](fig_lead_speedup.png)


## 5. Phonon Basis Scaling

These figures vary `n_phonon` while keeping the electronic lead size fixed. This isolates growth in the vibrational bath attached to the bridge-centered TTNS tree. It is useful because phonon modes increase both the local basis workload and the number of phonon Hamiltonian/product coupling terms.

The term and TTNO-bond plots show whether the operator representation grows mostly through flattened terms or through compact operator bonds. The timing plots then show how that representation growth appears in actual apply and construction costs.

![Phonon SOP terms](fig_phonon_terms.png)

![Phonon TTNO bond](fig_phonon_bond.png)

![Phonon apply time](fig_phonon_apply_time.png)

![Phonon construction time](fig_phonon_build_time.png)

![Phonon speedup](fig_phonon_speedup.png)


## 6. Operator-Count Stress Test

This is an artificial operator representation stress test, not a molecular-junction physical conclusion. It constructs `H = sum_ij V_ij A_i B_j` with low-rank coefficients and then stores it as a flattened SOP list. The purpose is to make product-term count grow rapidly while preserving shared structure that a compressed operator network can represent more economically.

The SOP-term plot verifies the intended product-count growth. The TTNO tensor-element plot checks whether compressed storage grows at the same rate. The speedup plot shows whether repeated flattened traversal becomes the dominant cost.

![Shared terms](fig_shared_terms.png)

![Shared TTNO tensor elements](fig_shared_ttno_elements.png)

![Shared speedup](fig_shared_speedup.png)


## 7. Interpretation

The SOP baseline applies each product term independently, mimicking the operator-layer behavior of conventional MCTDH/ML-MCTDH sum-of-products Hamiltonians. The TTNO representation produces numerically equivalent results while reusing common operator structure through tree-network bonds.

For the same symbolic Hamiltonian and TTNS state representation, the compressed TTNO operator layer is more efficient than the uncompressed flat SOP baseline in the tested basis-size and operator-count regimes. Small-system results should not be overinterpreted because TTNO construction and contraction overheads are visible, and simple Hamiltonians with only linear term growth may show a smaller advantage than correlated or highly repeated product-operator families.


## 8. Reproducibility

```bash
python benchmarks/benchmark_sop_vs_ttno.py --case lead --lead-list 1 2 4 8 16 32 --phonon 2 --repeats 7 --output benchmarks/results/lead_scaling.csv
python benchmarks/benchmark_sop_vs_ttno.py --case phonon --lead 4 --phonon-list 1 2 4 8 16 32 --repeats 7 --output benchmarks/results/phonon_scaling.csv
python benchmarks/benchmark_sop_vs_ttno.py --case shared-structure --size-list 4 8 16 32 64 --rank-list 1 2 4 --repeats 7 --output benchmarks/results/shared_structure_scaling.csv
python benchmarks/create_sop_vs_ttno_notebook.py
```
